## Setting up the API key

In [12]:
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import requests
import os

In [15]:
load_dotenv('env (1)', override=True)

weather_key = os.getenv('OPENWEATHER_API_KEY')
openrouter_key = os.getenv('OPENROUTER_API_KEY')

weather_key is not None, openrouter_key is not None

(True, True)

## To get live weather 

In [16]:
city = 'Manama'
url = 'https://api.openweathermap.org/data/2.5/weather'

querystring = {
    'q': city,
    'appid': weather_key,
    'units': 'metric'
}

response = requests.get(url, params=querystring)
weather = response.json()
weather

{'coord': {'lon': 50.5832, 'lat': 26.2154},
 'weather': [{'id': 800,
   'main': 'Clear',
   'description': 'clear sky',
   'icon': '01n'}],
 'base': 'stations',
 'main': {'temp': 34.66,
  'feels_like': 41.66,
  'temp_min': 34.66,
  'temp_max': 34.66,
  'pressure': 999,
  'humidity': 61,
  'sea_level': 999,
  'grnd_level': 998},
 'visibility': 10000,
 'wind': {'speed': 1.55, 'deg': 69, 'gust': 1.68},
 'clouds': {'all': 2},
 'dt': 1785954953,
 'sys': {'country': 'BH', 'sunrise': 1785895504, 'sunset': 1785943338},
 'timezone': 10800,
 'id': 290340,
 'name': 'Manama',
 'cod': 200}

In [17]:
live_weather = {
    'Date': str(pd.Timestamp.today().date()),
    'City': weather['name'],
    'Temperature': weather['main']['temp'],
    'Humidity': weather['main']['humidity'],
    'Wind Speed': round(weather['wind']['speed'] * 3.6, 2),
    'Condition': weather['weather'][0]['main']
}

live_weather

{'Date': '2026-08-05',
 'City': 'Manama',
 'Temperature': 34.66,
 'Humidity': 61,
 'Wind Speed': 5.58,
 'Condition': 'Clear'}

In [18]:
df = pd.read_csv('Data.csv')
new_weather = pd.DataFrame([live_weather])
df = pd.concat([df, new_weather], ignore_index=True)
df.to_csv('Data.csv', index=False)
df.tail()

,Date,City,Temperature,Humidity,Wind Speed,Condition
7,2026-07-05,Manama,25.00,50,10.00,Sunny
8,2026-07-05,Manama,25.00,50,10.00,Sunny
9,2026-07-05,Manama,25.00,50,10.00,Sunny
10,2026-08-04,Manama,25.00,50,10.00,Sunny
11,2026-08-05,Manama,34.66,61,5.58,Clear


In [ ]:
#from google.colab import userdata
#openai_api_key = userdata.get('OPENAI_API_KEY'),

## AI Meteorologist

In [20]:
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=openrouter_key)

In [21]:
def get_llm_response(prompt):
    completion = client.chat.completions.create(
        model='nvidia/nemotron-3-nano-30b-a3b:free',
        messages=[
            {'role': 'system', 'content': 'You are an AI meteorologist.'},
            {'role': 'user', 'content': prompt}
        ],
        temperature=0.0
    )
    return completion.choices[0].message.content

In [22]:
df = pd.read_csv('Data.csv')
last_7_days = df.tail(7)
last_7_days

,Date,City,Temperature,Humidity,Wind Speed,Condition
5,2025-07-05,Manama,25.00,50,10.00,Sunny
6,2026-07-05,Manama,25.00,50,10.00,Sunny
7,2026-07-05,Manama,25.00,50,10.00,Sunny
8,2026-07-05,Manama,25.00,50,10.00,Sunny
9,2026-07-05,Manama,25.00,50,10.00,Sunny
10,2026-08-04,Manama,25.00,50,10.00,Sunny
11,2026-08-05,Manama,34.66,61,5.58,Clear


In [23]:
prompt = f'''
Analyze this weather data:

{last_7_days}

Give a short weather summary, outfit suggestion,
and activity recommendation.
'''

response = get_llm_response(prompt)
response

'**Weather Summary**  \n- Mostly sunny with temperatures around **25\u202f°C** (77\u202f°F) and moderate humidity (~50%).  \n- A hotter, clearer day on **2026‑08‑05** reaches **34.7\u202f°C** (94\u202f°F) with slightly higher humidity (61%).  \n- Light wind (≈5–10\u202fkm/h) – generally calm.\n\n**Outfit Suggestion**  \n- **Daytime (25\u202f°C):** Light, breathable fabrics – a short‑sleeve shirt or tank top with shorts or a breezy skirt.  \n- **Hot day (34\u202f°C):** Switch to moisture‑wicking tops, loose linen or cotton pants/shorts, and a wide‑brim hat.  \n- Add **sunglasses** and **sunscreen (SPF\u202f30+)** for UV protection.  \n- If it’s windy, a light scarf can keep dust out of your face.\n\n**Activity Recommendation**  \n- **Morning/Evening outings:** Perfect for a stroll along the waterfront, a coffee at an outdoor café, or a light bike ride—temperatures are comfortable and the sun is gentle.  \n- **Mid‑day (especially on the 34\u202f°C day):** Opt for indoor or shaded activit